# 🎥 Drowsiness Detection — RUN

This notebook loads the **best trained checkpoint** and runs live webcam inference.

It automatically:
- Looks in `checkpoints/` for a usable model.
- Prefers `best.pth` (best validation accuracy). Falls back to `last.pth` if `best.pth` isn't there yet but training has made progress.
- If nothing usable is found, it stops and tells you exactly what to do (run `01_Train.ipynb`).

No need to edit anything below unless you want to change the webcam index or the decision threshold.

In [1]:
import os
import json
import cv2
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm

d:\GECACS\Dissertation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
CHECKPOINT_DIR = "checkpoints"
BEST_CKPT = os.path.join(CHECKPOINT_DIR, "best.pth")
LAST_CKPT = os.path.join(CHECKPOINT_DIR, "last.pth")
META_FILE = os.path.join(CHECKPOINT_DIR, "meta.json")

IMG_SIZE = 224
SEQ_LEN = 4
DROWSY_THRESHOLD = 0.65
WEBCAM_INDEX = 0

## Model definition

Must match the architecture used in training exactly.

In [4]:
class TemporalAttention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, lstm_out):
        scores = self.attention(lstm_out)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights * lstm_out, dim=1)
        return context


class CNN_LSTM_ViT(nn.Module):

    def __init__(self):
        super().__init__()
        mobilenet = torchvision.models.mobilenet_v3_large(weights=None)
        self.cnn = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        cnn_dim = 960

        self.lstm = nn.LSTM(
            input_size=cnn_dim, hidden_size=128, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3,
        )
        self.attention = TemporalAttention(128)

        self.vit = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=0)
        vit_dim = 192

        self.classifier = nn.Sequential(
            nn.Linear(256 + vit_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        cnn_input = x.view(B * T, C, H, W)
        cnn_features = self.cnn(cnn_input)
        cnn_features = self.pool(cnn_features).flatten(1)
        cnn_features = cnn_features.view(B, T, -1)

        lstm_out, _ = self.lstm(cnn_features)
        temporal_features = self.attention(lstm_out)

        last_frame = x[:, -1]
        vit_features = self.vit(last_frame)

        fusion = torch.cat([temporal_features, vit_features], dim=1)
        return self.classifier(fusion)

## Intelligent checkpoint selection

- `best.pth` exists → use it (best validation accuracy reached during training).
- otherwise, `last.pth` exists → use it, with a warning that it may not be the best epoch.
- otherwise → stop and print setup instructions.

In [5]:
def pick_checkpoint():
    if os.path.exists(BEST_CKPT):
        info = ""
        if os.path.exists(META_FILE):
            with open(META_FILE) as f:
                meta = json.load(f)
            info = f" (val_acc={meta.get('best_acc', '?'):.4f}, from {meta.get('epochs_completed','?')}/{meta.get('total_epochs','?')} epochs)"
        print(f"Using BEST checkpoint: {BEST_CKPT}{info}")
        state_dict = torch.load(BEST_CKPT, map_location=device)
        return state_dict

    if os.path.exists(LAST_CKPT):
        print(f"'best.pth' not found, but found '{LAST_CKPT}'. Using it (may not be the best-performing epoch).")
        ckpt = torch.load(LAST_CKPT, map_location=device)
        return ckpt["model_state"]

    raise FileNotFoundError(
        "\n\nNo trained model found.\n"
        "Expected one of:\n"
        f"  - {BEST_CKPT}\n"
        f"  - {LAST_CKPT}\n\n"
        "To fix this:\n"
        "  1. Open 01_Train.ipynb\n"
        "  2. Make sure your dataset paths in the Config cell are correct\n"
        "  3. Run all cells top to bottom\n"
        "  4. Training auto-saves checkpoints as it goes -- once at least one epoch\n"
        "     completes, 'checkpoints/best.pth' will exist and this notebook will work.\n"
        "  5. Come back here and re-run this notebook.\n"
    )


state_dict = pick_checkpoint()

model = CNN_LSTM_ViT().to(device)
model.load_state_dict(state_dict)
model.eval()
print("Model ready.")

Using BEST checkpoint: checkpoints\best.pth (val_acc=0.9903, from 10/10 epochs)
Model ready.


## Live webcam inference

Press **Esc** in the video window to stop.

In [ ]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Could not open webcam at index {WEBCAM_INDEX}. Try a different WEBCAM_INDEX.")

print("Webcam started. Press Esc in the video window to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))

    img = torch.tensor(rgb / 255.0, dtype=torch.float32).permute(2, 0, 1)
    img = normalize(img)

    seq = torch.stack([img] * SEQ_LEN).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(seq)
        prob = torch.softmax(output, dim=1)[0, 1].item()

    if prob > DROWSY_THRESHOLD:
        label, color = "DROWSY", (0, 0, 255)
    else:
        label, color = "ALERT", (0, 255, 0)

    cv2.putText(frame, f"{label} ({prob:.2f})", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.imshow("Driver Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Webcam started. Press Esc in the video window to stop.


KeyboardInterrupt: 

: 